# 00 · Data and EDA (Crop-CLIP)

**Plan:** `crop-clip/experiment-plan.md`, section 5.1 · **Task:** T1 + T2

What this notebook does:
1. Gets iWildCam v2.0 (team HF mirror of the official WILDS archive, pinned revision) and checks it:
   archive SHA-256, `metadata.csv` MD5, official split sizes, 182 classes, camera-disjoint splits.
2. Computes simple statistics for every image (size, brightness, colourfulness, infrared flag).
3. Draws the EDA figures: splits, class distribution, day vs infrared, cameras, bursts, preview grids.

Outputs (in `WORK_DIR`, also uploaded to W&B, project `team-support-vectors-idlf26/iwilds`,
group `crop-clip`): `image_stats.csv.gz` (artifact `iwildcam-image-stats`),
`species_cameras.csv`, `figures/eda_*.png`, `eda_summary.json`.

**Protocol:** preview grids show only **train / ID val / OOD val** photos. OOD-test photos are
never displayed; test label *counts* are fine (they are part of the dataset description).

**How to run (Molab):** run the cells top to bottom. The first run downloads ~12 GB and computes image
statistics (a few minutes with many CPU cores). Later runs reuse both.

In [ ]:
# Setup: shared code (crop-clip/common.py from the team repo) + packages + all imports
import importlib
import json
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/rnjema/idl-f26-g30-project.git"
REPO_BRANCH = "main"
REPO_DIR = Path(os.environ.get("CROPCLIP_REPO_DIR", Path.cwd() / "idl-f26-g30-project"))
PACKAGES = ["huggingface_hub", "wandb", "pandas", "matplotlib", "pillow", "tqdm"]

if (Path.cwd() / "common.py").is_file():  # notebook opened inside crop-clip/ of a clone
    CODE_DIR = Path.cwd()
else:  # Molab: clone (or update) the team repo next to the notebook
    if (REPO_DIR / ".git").is_dir():
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True
        )
    CODE_DIR = REPO_DIR / "crop-clip"
if os.environ.get("CROPCLIP_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)
sys.path.insert(0, str(CODE_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import common as cc

importlib.reload(cc)
print("shared code:", CODE_DIR / "common.py", "| git commit:", cc.git_commit(CODE_DIR))

## Settings and keys

Paste your W&B API key below **only in your private Molab copy**, or set `WANDB_API_KEY` as an
environment variable. Put the placeholder back before sharing or committing the notebook.

In [ ]:
WANDB_API_KEY = "PASTE_YOUR_WANDB_API_KEY_HERE"  # private copy only; never commit a real key
USE_WANDB = os.environ.get("CROPCLIP_USE_WANDB", "1") == "1"
WANDB_BASE_URL = "https://api.forge.coreweave.com"  # team W&B server (CoreWeave Forge)
IMAGE_WORKERS = max(1, (os.cpu_count() or 2) - 1)  # processes for the image statistics
SEED = 0

if "PASTE_YOUR" not in WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_BASE_URL"] = WANDB_BASE_URL
cc.WANDB_BASE_URL = WANDB_BASE_URL  # common.py uses it for every run and artifact
if USE_WANDB and not os.environ.get("WANDB_API_KEY"):
    print("No W&B key found: running without W&B (figures are still saved locally).")
    USE_WANDB = False

WORK_DIR = cc.work_root()
FIG_DIR = WORK_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("data root:", cc.data_root(), "| work dir:", WORK_DIR, "| W&B:", USE_WANDB)

## 1 · Dataset: download (first run only) and verify

In [ ]:
DS_DIR = cc.ensure_dataset()
meta, CLASS_NAMES = cc.load_metadata(DS_DIR)
PROTOCOL = cc.verify_protocol(meta)
print(f"{len(meta):,} images | {len(CLASS_NAMES)} classes (y=0 is {CLASS_NAMES[0]!r})")
print("split sizes:", PROTOCOL["split_sizes"])
print("cameras per camera-disjoint split:", PROTOCOL["ood_camera_counts"], "| no camera shared")
meta.head()

In [ ]:
# W&B run for this notebook + a helper that saves every figure locally and to W&B
RUN = cc.wandb_run(
    "cropclip-00-eda",
    "eda",
    {"notebook": "00_data_eda", "n_images": len(meta), "mirror_revision": cc.MIRROR_REVISION},
    enabled=USE_WANDB,
)


def save_fig(fig, name):
    path = FIG_DIR / f"eda_{name}.png"
    fig.savefig(path, dpi=120, bbox_inches="tight")
    if RUN is not None:
        import wandb

        RUN.log({f"eda/{name}": wandb.Image(str(path))})
    plt.close(fig)
    return fig

## 2 · Per-image statistics

For each photo: width, height, brightness, *colourfulness* (mean difference between the strongest
and weakest colour channel) and an **infrared flag** (infrared night frames are grayscale, so their
colourfulness is near 0). Computed once (reused from disk or from the W&B artifact) at 1/8 scale.

In [ ]:
_stats_path = WORK_DIR / "image_stats.csv.gz"
if not _stats_path.is_file() and USE_WANDB:
    try:  # a teammate may already have computed it
        cc.fetch_artifact("iwildcam-image-stats", WORK_DIR, run=RUN)
    except Exception as _err:
        print("no image-stats artifact yet:", type(_err).__name__)
if _stats_path.is_file():
    img_stats = pd.read_csv(_stats_path)
    print("loaded", _stats_path)
else:
    _paths = [cc.image_path(DS_DIR, _f) for _f in meta["filename"]]
    img_stats = cc.compute_image_stats(_paths, workers=IMAGE_WORKERS)
    img_stats.insert(0, "filename", meta["filename"].to_numpy())
    img_stats.to_csv(_stats_path, index=False)
    cc.log_artifact(
        RUN,
        "iwildcam-image-stats",
        "dataset-stats",
        [_stats_path],
        {"ir_colourfulness_max": cc.IR_COLOURFULNESS_MAX, "n_images": len(img_stats)},
    )
assert len(img_stats) == len(meta) and set(img_stats["filename"]) == set(meta["filename"])
meta_img = meta.merge(img_stats, on="filename", how="left", validate="one_to_one")
print(
    f"readable {int(meta_img['readable'].sum()):,}/{len(meta_img):,} | "
    f"infrared {100 * meta_img['ir'].mean():.1f}%"
)

In [ ]:
# Check the infrared rule: colourfulness should split into a spike near 0 (IR) and a broad hump (day)
_fig, _ax = plt.subplots(figsize=(9, 3.5))
_ax.hist(meta_img["colourfulness"].dropna().clip(upper=60), bins=120, color="0.4")
_ax.axvline(cc.IR_COLOURFULNESS_MAX, color="C3", ls="--", label=f"IR threshold {cc.IR_COLOURFULNESS_MAX}")
_ax.set_yscale("log")
_ax.set_xlabel("colourfulness (mean max-min over RGB, 0-255 scale; clipped at 60)")
_ax.set_ylabel("photos (log)")
_ax.set_title("Infrared detection: photos left of the red line are flagged as infrared")
_ax.legend()
save_fig(_fig, "ir_threshold")

## 3 · Splits overview

Bursts = camera sequences (one trigger fires several frames). OOD val and OOD test are the
camera-disjoint evaluation splits; ID val / ID test reuse training cameras.

In [ ]:
SPLIT_TABLE = cc.split_overview(meta_img)
if RUN is not None:
    import wandb as _wandb

    RUN.log({"eda/split_overview": _wandb.Table(dataframe=SPLIT_TABLE)})
SPLIT_TABLE

## 4 · Class distribution (the long tail)

Macro F1 averages the classes present in a split **equally**, so a species with 5 test photos
counts as much as one with thousands.

In [ ]:
SPECIES = cc.species_camera_table(meta, CLASS_NAMES)
SPECIES.to_csv(WORK_DIR / "species_cameras.csv", index=False)
_s = SPECIES.sort_values("train_images", ascending=False).reset_index(drop=True)
_in_test = _s["test_images"] > 0
_fig, _ax = plt.subplots(figsize=(12, 4))
_ax.bar(range(len(_s)), _s["train_images"] + 0.5, width=1.0, color=np.where(_in_test, "C0", "0.75"))
_ax.set_yscale("log")
_ax.set_xlabel("class rank (by training images)")
_ax.set_ylabel("training images (log)")
_ax.set_title(
    f"Training images per class: blue = present in OOD test ({int(_in_test.sum())} classes), "
    f"grey = not in OOD test"
)
save_fig(_fig, "class_long_tail")

In [ ]:
# Top 20 classes per split (label counts)
_fig, _axes = plt.subplots(1, 3, figsize=(16, 6))
for _ax, _split in zip(_axes, ("train", "val", "test")):
    _counts = meta.loc[meta["split"] == _split, "class_name"].value_counts().head(20)[::-1]
    _ax.barh(_counts.index, _counts.to_numpy(), color="C0")
    _ax.set_title(f"{cc.SPLIT_LABELS[_split]}: top 20 classes")
    _ax.tick_params(axis="y", labelsize=8)
_fig.tight_layout()
save_fig(_fig, "top_classes_per_split")

In [ ]:
# How many training images do the classes in each evaluation split have?
_bins = [0, 1, 10, 100, 1000, 10**9]
_labels = ["0", "1-9", "10-99", "100-999", "1000+"]
_rows = []
for _split in ("id_val", "val", "id_test", "test"):
    _present = SPECIES[SPECIES[f"{_split}_images"] > 0]
    _binned = pd.cut(_present["train_images"], _bins, right=False, labels=_labels)
    _rows.append({"split": cc.SPLIT_LABELS[_split], **_binned.value_counts().reindex(_labels).to_dict()})
TAIL_TABLE = pd.DataFrame(_rows).set_index("split")
TAIL_TABLE.columns = [f"classes with {c} train images" for c in TAIL_TABLE.columns]
TAIL_TABLE

## 5 · Image setting: day vs infrared, time of day, brightness, image size

In [ ]:
_fig, _axes = plt.subplots(1, 3, figsize=(17, 4))
_order = list(cc.SPLITS)
_ir = meta_img.groupby("split")["ir"].mean().reindex(_order) * 100
_axes[0].bar([cc.SPLIT_LABELS[s] for s in _order], _ir.to_numpy(), color="C1")
_axes[0].set_ylabel("% infrared photos")
_axes[0].set_title("Infrared (night) share per split")
for _split in ("train", "val", "test"):
    _h = meta_img.loc[meta_img["split"] == _split, "hour"].value_counts(normalize=True).sort_index()
    _axes[1].plot(_h.index, 100 * _h.to_numpy(), marker="o", ms=3, label=cc.SPLIT_LABELS[_split])
    _b = meta_img.loc[meta_img["split"] == _split, "brightness"].dropna()
    _axes[2].hist(_b, bins=60, density=True, histtype="step", label=cc.SPLIT_LABELS[_split])
_axes[1].set_xlabel("hour of day")
_axes[1].set_ylabel("% of photos")
_axes[1].set_title("Photos per hour")
_axes[1].legend()
_axes[2].set_xlabel("mean brightness (0-255)")
_axes[2].set_title("Brightness")
_axes[2].legend()
_fig.tight_layout()
save_fig(_fig, "day_night_brightness")

In [ ]:
# Image sizes (top 8) and infrared share by split for animal vs empty photos
SIZE_TABLE = (
    meta_img.groupby(["width", "height"]).size().sort_values(ascending=False).head(8).rename("photos")
)
print(SIZE_TABLE.to_string())
IR_BY_LABEL = (
    meta_img.assign(kind=np.where(meta_img["is_empty"], "empty", "animal"))
    .pivot_table(index="split", columns="kind", values="ir", aggfunc="mean")
    .reindex(list(cc.SPLITS))
    .mul(100)
    .round(1)
)
IR_BY_LABEL.columns = [f"% infrared ({c})" for c in IR_BY_LABEL.columns]
IR_BY_LABEL

## 6 · Cameras

The EDA's strongest predictor of failure: a species photographed at only one or two **training
cameras** gives the model one or two backgrounds to learn from.

In [ ]:
_fig, _axes = plt.subplots(1, 3, figsize=(17, 4))
for _ax, _split in zip(_axes, ("train", "val", "test")):
    _per_cam = meta.loc[meta["split"] == _split, "camera"].value_counts().to_numpy()
    _ax.bar(range(len(_per_cam)), _per_cam, width=1.0, color="C2")
    _ax.set_yscale("log")
    _ax.set_title(f"{cc.SPLIT_LABELS[_split]}: photos per camera ({len(_per_cam)} cameras)")
    _ax.set_xlabel("camera rank")
_fig.tight_layout()
save_fig(_fig, "photos_per_camera")

In [ ]:
CAMERA_TABLE = (
    meta[meta["split"].isin(["train", "val", "test"])]
    .groupby(["split", "camera"])
    .agg(photos=("y", "size"), species=("y", lambda s: s[s != cc.EMPTY_Y].nunique()), empty_rate=("is_empty", "mean"))
    .reset_index()
)
_fig, _axes = plt.subplots(1, 2, figsize=(13, 4))
for _split in ("train", "val", "test"):
    _part = CAMERA_TABLE[CAMERA_TABLE["split"] == _split]
    _axes[0].hist(_part["species"], bins=range(0, 45), histtype="step", density=True, label=cc.SPLIT_LABELS[_split])
    _axes[1].hist(_part["empty_rate"], bins=20, range=(0, 1), histtype="step", density=True, label=cc.SPLIT_LABELS[_split])
_axes[0].set_xlabel("species per camera")
_axes[0].set_title("How many species each camera sees")
_axes[1].set_xlabel("share of empty photos at the camera")
_axes[1].set_title("Empty rate per camera")
for _ax in _axes:
    _ax.legend()
_fig.tight_layout()
save_fig(_fig, "camera_species_empty")

In [ ]:
# Training cameras per species, for the species scored in OOD test
_scored = SPECIES[(SPECIES["test_images"] > 0) & (SPECIES["y"] != cc.EMPTY_Y)]
_fig, _axes = plt.subplots(1, 2, figsize=(13, 4))
_axes[0].hist(_scored["train_cameras"], bins=range(0, int(_scored["train_cameras"].max()) + 2), color="C4")
_axes[0].set_xlabel("number of training cameras the species was seen at")
_axes[0].set_ylabel("species in OOD test")
_axes[0].set_title("Training cameras per OOD-test species")
_few = _scored["train_cameras"] <= 2
_axes[1].scatter(_scored["train_images"] + 0.5, _scored["train_cameras"], c=np.where(_few, "C3", "C0"), s=14)
_axes[1].set_xscale("log")
_axes[1].set_xlabel("training images (log)")
_axes[1].set_ylabel("training cameras")
_axes[1].set_title("Red = seen at only 1-2 training cameras")
_fig.tight_layout()
print(
    f"{int(_few.sum())} of {len(_scored)} OOD-test species were seen at <= 2 training cameras; "
    f"{int((_scored['train_cameras'] == 0).sum())} never appear in training."
)
save_fig(_fig, "train_cameras_per_species")

## 7 · Bursts (camera sequences)

In [ ]:
_burst = meta.groupby(["split", "burst"]).agg(frames=("y", "size"), labels=("y", "nunique")).reset_index()
_fig, _ax = plt.subplots(figsize=(9, 3.5))
for _split in ("train", "val", "test"):
    _f = _burst.loc[_burst["split"] == _split, "frames"].clip(upper=15)
    _ax.hist(_f, bins=range(1, 17), histtype="step", density=True, label=cc.SPLIT_LABELS[_split])
_ax.set_xlabel("frames per burst (clipped at 15)")
_ax.set_ylabel("share of bursts")
_ax.set_title("Burst length")
_ax.legend()
BURST_TABLE = (
    _burst.groupby("split")
    .agg(bursts=("burst", "size"), mean_frames=("frames", "mean"), single_label_share=("labels", lambda s: (s == 1).mean()))
    .reindex(list(cc.SPLITS))
    .round(3)
)
print(BURST_TABLE.to_string())
save_fig(_fig, "burst_length")

## 8 · Preview grids (train / ID val / OOD val only)

Each tile shows: class, split, camera, date/time, DAY or IR.

In [ ]:
def show_grid(rows, title, name, ncols=4, tile=3.4):
    rows = rows.reset_index(drop=True)
    nrows = max(1, int(np.ceil(len(rows) / ncols)))
    fig, axes = plt.subplots(nrows, ncols, figsize=(tile * ncols, tile * 0.8 * nrows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), rows.iterrows()):
        with Image.open(cc.image_path(DS_DIR, r["filename"])) as img:
            img.draft("RGB", (img.width // 2, img.height // 2))
            ax.imshow(img.convert("RGB"))
        mode = "IR" if r.get("ir", False) else "DAY"
        ax.set_title(
            f"{r['class_name'][:28]}\n{cc.SPLIT_LABELS[r['split']]} · cam {r['camera']} · "
            f"{str(r['when'])[:16]} · {mode}",
            fontsize=7,
        )
    fig.suptitle(title, fontsize=11)
    fig.tight_layout()
    return save_fig(fig, name)


def pick(rows, n):
    # up to n random rows (fewer if the group is small)
    return rows.sample(min(n, len(rows)), random_state=SEED)


PREVIEW_POOL = meta_img[meta_img["split"].isin(["train", "id_val", "val"])]
TRAIN_POOL = PREVIEW_POOL[PREVIEW_POOL["split"] == "train"]
print("preview pool:", PREVIEW_POOL["split"].value_counts().to_dict())

In [ ]:
_sample = pd.concat(
    [pick(PREVIEW_POOL[PREVIEW_POOL["split"] == _s], 4) for _s in ("train", "id_val", "val")]
)
show_grid(_sample, "Random photos: train, ID val, OOD val", "grid_random", ncols=4)

In [ ]:
_animals = TRAIN_POOL[~TRAIN_POOL["is_empty"]]
_sample = pd.concat([pick(_animals[~_animals["ir"]], 4), pick(_animals[_animals["ir"]], 4)])
show_grid(_sample, "Training animals: daylight (top) vs infrared night (bottom)", "grid_day_ir", ncols=4)

In [ ]:
_top = TRAIN_POOL.loc[~TRAIN_POOL["is_empty"], "y"].value_counts().head(6).index
_sample = pd.concat([pick(TRAIN_POOL[TRAIN_POOL["y"] == _y], 2) for _y in _top])
show_grid(_sample, "Most common training species (2 photos each)", "grid_common", ncols=4)

In [ ]:
_empties = TRAIN_POOL[TRAIN_POOL["is_empty"]]
_sample = pd.concat([pick(_empties[~_empties["ir"]], 4), pick(_empties[_empties["ir"]], 4)])
show_grid(_sample, "Training photos labelled EMPTY: day (top) vs infrared (bottom)", "grid_empty", ncols=4)

In [ ]:
# Species seen at only 1-2 training cameras: everything the model gets to learn from them
_rare_species = SPECIES[(SPECIES["train_cameras"].between(1, 2)) & (SPECIES["y"] != cc.EMPTY_Y)]
_rare_species = _rare_species.sort_values("train_images").head(8)
_sample = pd.concat([pick(TRAIN_POOL[TRAIN_POOL["y"] == _y], 1) for _y in _rare_species["y"]])
show_grid(_sample, "Species seen at only 1-2 training cameras (one training photo each)", "grid_few_cameras", ncols=4)

## 9 · Summary

Headline numbers for the report, saved to `eda_summary.json` and the W&B run summary.

In [ ]:
_scored_species = SPECIES[(SPECIES["test_images"] > 0) & (SPECIES["y"] != cc.EMPTY_Y)]
EDA_SUMMARY = {
    "n_images": int(len(meta_img)),
    "pct_infrared_by_split": {s: round(100 * float(meta_img.loc[meta_img["split"] == s, "ir"].mean()), 1) for s in cc.SPLITS},
    "pct_empty_by_split": {s: round(100 * float(meta_img.loc[meta_img["split"] == s, "is_empty"].mean()), 1) for s in cc.SPLITS},
    "classes_in_ood_test": int((SPECIES["test_images"] > 0).sum()),
    "ood_test_species_at_le2_train_cameras": int((_scored_species["train_cameras"] <= 2).sum()),
    "single_label_burst_share_train": float(BURST_TABLE.loc["train", "single_label_share"]),
    "ir_colourfulness_max": cc.IR_COLOURFULNESS_MAX,
    "git_commit": cc.git_commit(CODE_DIR),
}
(WORK_DIR / "eda_summary.json").write_text(json.dumps(EDA_SUMMARY, indent=2))
if RUN is not None:
    RUN.summary.update({f"eda/{k}": v for k, v in EDA_SUMMARY.items() if not isinstance(v, dict)})
    RUN.finish()
EDA_SUMMARY